# Selectivity Sweep - Correlation Verification
This notebook verifies that the synthetic metadata scrambling (derangement) hit the target correlation percentages (100%, 75%, 50%, 25%, 0%) for the `hm_clothes` tables.

It reads the original categories directly from the benchmark payload file and cross-references them with the actual stored categories in each isolated PostgreSQL table.

**Additionally**, it introduces a spatial manipulation-check using **k-NN label purity** on a sample of the dataset. This provides empirical evidence that our target scrambling percentages actually produced the intended spatial effect within the pgvector index, validating the operationalized independent variable.


In [ ]:
import json
import os
import psycopg2
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

conn = psycopg2.connect(
    dbname=os.getenv("POSTGRES_DB", "vector_db"),
    user=os.getenv("POSTGRES_USER", "postgres"),
    password=os.getenv("POSTGRES_PASSWORD", "mysecretpassword"),
    host=os.getenv("POSTGRES_HOST", "127.0.0.1"),
    port=os.getenv("POSTGRES_HOST_PORT", "5433"),
)

data_dir = os.path.join("ann-filtering-benchmark-datasets", "data")
payloads_path = os.path.join(data_dir, "payloads.jsonl")

target_columns = [
    "index_group_name",
    "product_type_name",
    "garment_group_name",
    "perceived_colour_master_name",
    "graphical_appearance_name"
]

original_metadata = []
with open(payloads_path, encoding="utf-8") as f:
    for line in f:
        payload = json.loads(line)
        original_metadata.append(tuple(payload.get(c, "Unknown") for c in target_columns))
        
print(f"Loaded {len(original_metadata)} original metadata bundles.")


In [ ]:
tables = ["hm_clothes_100", "hm_clothes_75", "hm_clothes_50", "hm_clothes_25", "hm_clothes_0"]
results = []

for table in tables:
    query = f"SELECT vector_id, index_group_name, product_type_name, garment_group_name, perceived_colour_master_name, graphical_appearance_name FROM {table} ORDER BY vector_id ASC"
    df_db = pd.read_sql(query, conn)
    db_metadata = list(df_db.itertuples(index=False, name=None))
    db_bundles = [tup[1:] for tup in db_metadata] # skip vector_id

    matches = sum(1 for orig, db in zip(original_metadata, db_bundles) if orig == db)
    match_percentage = (matches / len(original_metadata)) * 100

    results.append({
        "Table": table,
        "Target Correlation (%)": int(table.split("_")[-1]),
        "Actual Correlation (%)": round(match_percentage, 2),
        "Exact Bundle Matches": matches,
    })

results_df = pd.DataFrame(results)
display(results_df)


## k-NN Spatial Purity Check

To prove our scrambling successfully affected spatial layout in the vector space, we calculate the k-NN purity on a sample of 2,000 vectors for each correlation tier. 

For each vector in the sample, we retrieve its `k=10` nearest neighbors based on vector distance, and calculate the fraction of those neighbors that share the **exact same 5-column metadata bundle** as the query vector. 

*If our manipulation is successful, the average k-NN purity should strictly decrease from `hm_clothes_100` down to `hm_clothes_0`.*


In [ ]:
k_neighbors = 10
sample_size = 2000
purity_results = []

# Fetch a random sample of vector_ids to test across all tables consistently
sample_query = "SELECT vector_id FROM hm_clothes_100 ORDER BY RANDOM() LIMIT %s"
sample_ids = pd.read_sql(sample_query, conn, params=(sample_size,))["vector_id"].tolist()

print(f"Running k-NN purity check for {sample_size} sample vectors (k={k_neighbors})...")

for table in tables:
    purity_scores = []
    
    with conn.cursor() as cur:
        for vid in sample_ids:
            # Get the query vector and its metadata bundle
            cur.execute(f"""
                SELECT embedding, index_group_name, product_type_name, garment_group_name, perceived_colour_master_name, graphical_appearance_name 
                FROM {table} 
                WHERE vector_id = %s
            """, (vid,))
            row = cur.fetchone()
            if not row: continue
            
            query_emb = row[0]
            query_bundle = row[1:]
            
            # Find k nearest neighbors using pgvector
            cur.execute(f"""
                SELECT index_group_name, product_type_name, garment_group_name, perceived_colour_master_name, graphical_appearance_name
                FROM {table}
                WHERE vector_id != %s
                ORDER BY embedding <-> %s
                LIMIT %s
            """, (vid, query_emb, k_neighbors))
            
            neighbors = cur.fetchall()
            
            # Purity: how many neighbors share the exact same bundle?
            matches = sum(1 for n in neighbors if tuple(n) == query_bundle)
            purity_scores.append(matches / k_neighbors)
            
    avg_purity = np.mean(purity_scores)
    
    purity_results.append({
        "Table": table,
        "Target Correlation (%)": int(table.split("_")[-1]),
        "Average k-NN Purity": round(avg_purity, 4)
    })

purity_df = pd.DataFrame(purity_results)
display(purity_df)

# Plot
plt.figure(figsize=(8, 5))
plt.plot(purity_df["Target Correlation (%)"], purity_df["Average k-NN Purity"], marker='o', linestyle='-')
plt.title(f"Spatial Label Purity vs. Scrambling Target (k={k_neighbors})")
plt.xlabel("Target Bundle-Row Correlation (%)")
plt.ylabel("Average k-NN Bundle Purity")
plt.grid(True)
plt.gca().invert_xaxis() # 100 to 0
plt.show()
